# 05 — Rapier 3D Simulation

This notebook uses the full Rapier physics engine (Rust rigid-body simulation) instead of the fast analytical model.

You get:
- True 3D rope positions at every timestep
- Per-link force data
- `SimulationReplay` for frame-by-frame access
- Matplotlib animation of the fall

The simulation runs in Rust; Python only handles setup and visualisation.

In [ ]:
import ropesim.notebook
from ropesim import Rope, Scenario, BelayDevice, PhysicsMode
from ropesim.replay import SimulationReplay
from ropesim._rustcore import PyRopeSimWorld
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

## 1. Build a Rapier world directly

`PyRopeSimWorld` is the raw Rust physics world. Here we build a 15 m rope with an 80 kg climber attached to the bottom link, then let it fall.

In [ ]:
# Create world with 9.81 m/s^2 gravity
world = PyRopeSimWorld(9.81)

# Add a 15-metre rope: start at top (anchor), end at bottom (climber attach)
# Parameters: start_pos, end_pos, length_m, mass_per_meter, link_m, stiffness_kn, damping
rope_handles = world.add_rope(
    [0.0, 15.0, 0.0],   # anchor at top
    [0.0,  0.0, 0.0],   # climber at bottom
    15.0,               # rope length (m)
    0.065,              # mass per metre (kg/m)
    0.08,               # link length (m) -- ~188 links
    40.0,               # stiffness (kN)
    0.12,               # damping ratio
)
print(f'Rope links: {len(rope_handles)}')

# Attach 80 kg climber to bottom rope link
climber_handle = world.add_climber(rope_handles[-1], 80.0)
print('Climber attached to bottom rope link')

## 2. Run the simulation

`step_n(n, dt)` runs `n` timesteps of `dt` seconds each in Rust and returns a `SimFrameData` object containing all positions and forces.

In [ ]:
# 1 second of simulation at 1/240 s timestep = 240 steps
DT = 1.0 / 240.0
N_STEPS = 240

print(f'Simulating {N_STEPS} steps ({N_STEPS*DT:.3f}s) ...')
frame_data = world.step_n(N_STEPS, DT)
print(f'Done. Frames: {len(frame_data.frames)}')
print(f'Peak anchor force: {frame_data.peak_anchor_force():.3f} kN')
print(f'Peak deceleration: {frame_data.peak_deceleration_g():.2f} g')

## 3. Explore with SimulationReplay

In [ ]:
replay = SimulationReplay(frame_data)

print(f'Total frames:       {replay.total_frames()}')
print(f'Peak force frame:   {replay.peak_force_frame()}')
print(f'First catch frame:  {replay.first_catch_frame()}')
print(f'Simulation time:    {replay.total_time_seconds:.3f} s')

# Inspect a specific frame
peak_frame = replay[replay.peak_force_frame()]
print(f'\nPeak frame:')
print(f'  t = {peak_frame.timestamp_ms:.1f} ms')
print(f'  anchor force = {peak_frame.anchor_force_kn:.3f} kN')
print(f'  climber pos  = [{peak_frame.climber_position[0]:.2f}, {peak_frame.climber_position[1]:.2f}, {peak_frame.climber_position[2]:.2f}]')

## 4. Plot the 3D anchor force curve

In [ ]:
forces = replay.force_curve()
t_ms   = [frame_data.frames[i].timestamp_ms for i in range(len(forces))]

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(t_ms, forces, color='#2270c4', linewidth=1.8)
ax.fill_between(t_ms, forces, alpha=0.15, color='#2270c4')

pf_idx = replay.peak_force_frame()
ax.scatter([t_ms[pf_idx]], [forces[pf_idx]], color='red', s=60, zorder=5,
           label=f'Peak: {forces[pf_idx]:.3f} kN')

fc_idx = replay.first_catch_frame()
ax.axvline(t_ms[fc_idx], color='green', linestyle='--', linewidth=0.8,
           label=f'Rope taut @ {t_ms[fc_idx]:.0f} ms')

ax.set_xlabel('Time (ms)')
ax.set_ylabel('Anchor force (kN)')
ax.set_title('Rapier 3D simulation — anchor force vs time')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 5. 3D snapshot of rope positions

Each `SimFrame` contains the full 3D position of every rope link. We can visualise the rope shape at any instant.

In [ ]:
def plot_rope_at_frame(idx: int, ax=None, colour='#2270c4', alpha=1.0, label=''):
    frame = replay[idx]
    pos = np.array(frame.rope_positions)  # shape (n_links, 3)
    climber = np.array(frame.climber_position)
    if ax is None:
        fig = plt.figure(figsize=(4, 7))
        ax = fig.add_subplot(111, projection='3d')
    ax.plot(pos[:, 0], pos[:, 2], pos[:, 1],
            color=colour, linewidth=2, alpha=alpha, label=label)
    ax.scatter(*[climber[i] for i in [0, 2, 1]], color='red', s=80, zorder=5)
    return ax

# Plot rope at start, mid-fall, and peak force moment
fig = plt.figure(figsize=(12, 6))
titles = [
    (0,                           'Start (t=0)'),
    (replay.first_catch_frame(),  'Rope goes taut'),
    (replay.peak_force_frame(),   'Peak force'),
    (N_STEPS - 1,                 'End of sim'),
]
for i, (fidx, title) in enumerate(titles):
    ax = fig.add_subplot(1, 4, i+1, projection='3d')
    plot_rope_at_frame(fidx, ax=ax)
    ax.set_title(f'{title}\nt={frame_data.frames[fidx].timestamp_ms:.0f}ms', fontsize=9)
    ax.set_xlabel('X', fontsize=7)
    ax.set_ylabel('Z', fontsize=7)
    ax.set_zlabel('Y (height)', fontsize=7)
    ax.tick_params(labelsize=6)
plt.suptitle('Rope shape at key simulation moments', fontsize=11)
plt.tight_layout()
plt.show()

## 6. Climber trajectory

In [ ]:
cx = [f.climber_position[0] for f in frame_data.frames]
cy = [f.climber_position[1] for f in frame_data.frames]  # height
cz = [f.climber_position[2] for f in frame_data.frames]
vy = [f.climber_velocity[1] for f in frame_data.frames]   # vertical velocity
t  = [f.timestamp_ms for f in frame_data.frames]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(t, cy, color='#27ae60', linewidth=2)
ax1.set_xlabel('Time (ms)')
ax1.set_ylabel('Height (m)')
ax1.set_title('Climber height vs time')
ax1.axhline(0, color='grey', linestyle='--', linewidth=0.7, label='Ground level')
ax1.legend(fontsize=8)

ax2.plot(t, vy, color='#e74c3c', linewidth=2)
ax2.axhline(0, color='grey', linestyle='--', linewidth=0.7)
ax2.set_xlabel('Time (ms)')
ax2.set_ylabel('Vertical velocity (m/s)')
ax2.set_title('Climber vertical velocity vs time')
ax2.fill_between(t, vy, where=[v < 0 for v in vy], alpha=0.2, color='red', label='Falling')
ax2.fill_between(t, vy, where=[v >= 0 for v in vy], alpha=0.2, color='green', label='Bouncing up')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 7. Use PhysicsMode.RAPIER_3D from Scenario

For the full integrated API, pass `mode=PhysicsMode.RAPIER_3D` to `scenario.simulate_fall()`.

In [ ]:
rope = Rope.from_db('Beal Opera 8.5 Dry')
scenario = Scenario(rope, climber_mass_kg=80.0)

# ANALYTICAL is the default (fast)
result_analytical = scenario.simulate_fall(10.0, mode=PhysicsMode.ANALYTICAL)
print(f'Analytical:  {result_analytical.peak_force_kn:.3f} kN')

# RAPIER_3D: full simulation via the Rust world
result_rapier = scenario.simulate_fall(10.0, mode=PhysicsMode.RAPIER_3D, dt_seconds=1/240)
print(f'Rapier 3D:   {result_rapier.peak_force_kn:.3f} kN')

print(f'\nAnalytical vs Rapier: {abs(result_analytical.peak_force_kn - result_rapier.peak_force_kn):.3f} kN difference')

## 8. Async simulation in a notebook

For parameter sweeps in async code (or Jupyter with an async kernel):

In [ ]:
import asyncio

async def run_async_sweep():
    results = await scenario.sweep_fall_positions_async(
        height_range=(5.0, 25.0), steps=30
    )
    print(f'Async sweep complete: {len(results.peak_forces_kn)} positions')
    print(f'Worst-case: {results.worst_height_m:.1f}m -> {results.worst_peak_kn:.2f} kN')
    return results

# In a standard (non-async) notebook kernel:
results = asyncio.run(run_async_sweep())

## Key takeaways

- `PyRopeSimWorld` gives you direct access to the Rapier simulation: add bodies, step manually, query anything.
- `step_n(n, dt)` pre-simulates the full fall in one call and stores all frames — use `SimulationReplay` to access them.
- The Rapier result and the analytical UIAA result should be close for clean lead falls; Rapier adds value for complex scenarios (pendulum, rock contact, dynamic belayer).
- All Rapier computation happens in Rust; Python only handles setup and visualisation, so performance is excellent.